# Experiment: 5D cond 1D

dim(x)=4, dim(y)=1 — comparing LGD vs LGD-CM.

In [1]:
# ============================================================
# CONFIG — only this cell changes between notebooks
# ============================================================
EXPERIMENT_NAME   = "5D_cond_1D"
GLOBAL_SEED       = 42
FORCE_RETRAIN     = False

BASE_DIR          = "/content/conditional-matching-paper/simulations"
PARAMS_DIR        = f"{BASE_DIR}/params"
CHECKPOINT_DIR    = f"{BASE_DIR}/checkpoints/{EXPERIMENT_NAME}"
RESULTS_DIR       = f"{BASE_DIR}/results/{EXPERIMENT_NAME}"

# Architecture — Diffusion models
NBLOCKS           = 6
NUNITS            = 512

# Architecture — Consistency Model
NBLOCKS_CM        = 6
NUNITS_CM         = 512

# Training — Diffusion
NEPOCHS           = 40_000#20_000
BATCH_SIZE        = 4_096#512

# Training — Consistency Model
NEPOCHS_CM        = 40_000
BATCH_SIZE_CM     = 4_096

# Diffusion
DIFFUSION_STEPS   = 150

# Optimization
N_ATTEMP_OPTIM              = 25
NSAMPLES_IN_OPTIM_FOR_MMD   = 250
NUM_X_T_LGD                 = 3
NUM_X_T_LGD_CM              = 3

# GMM dimensions
CONDITION_ON      = 4   # dim(x)=4, dim(y)=1

In [2]:
!pip install flow_matching -q
!pip install POT -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.0 MB/s eta 0:00:00


In [3]:
import os, sys
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

if 'google.colab' in str(get_ipython()):
    import getpass
    !pip install -q diffusers transformers accelerate xformers
    !pip install -q scikit-learn matplotlib Pillow

    github_token = userdata.get('GITHUB')
    token = github_token if github_token else getpass.getpass("Enter your GitHub personal access token: ")

    repo_url  = f"https://{token}@github.com/orineo1/conditional-matching-paper.git"
    repo_name = "conditional-matching-paper"
    branch    = "adding-simu-compare"

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    !cd {repo_name} && git checkout {branch}

# ── point Python at simulations/src where all .py modules live ──
src_path = f"/content/{repo_name}/simulations/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Branch: {branch}")
print(f"src path on sys.path: {src_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.4 MB/s eta 0:00:00
Cloning into 'conditional-matching-paper'...
remote: Enumerating objects: 6265, done.
remote: Counting objects: 100% (376/376), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 6265 (delta 364), reused 291 (delta 291), pack-reused 5889 (from 3)
Receiving objects: 100% (6265/6265), 1.28 GiB | 17.35 MiB/s, done.
Resolving deltas: 100% (1720/1720), done.
Updating files: 100% (255/255), done.
error: pathspec 'adding-simu-compare' did not match any file(s) known to git
Branch: adding-simu-compare
src path on sys.path: /content/conditional-matching-paper/simulations/src


In [4]:
import os, sys, time, json
import importlib
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial
from tqdm import trange

import Diffusion
import LossFunctions
import ConsistencyModels
import dist_utils
import Optimization
import experiment_utils
from ConsistencyModels import ConsistencyModeliCT
import evalModels

for mod in [Diffusion, LossFunctions, ConsistencyModels,
            dist_utils, Optimization, experiment_utils]:
    importlib.reload(mod)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)
os.makedirs(PARAMS_DIR,     exist_ok=True)
print("Imports done.")

Imports done.


In [5]:
env_info = experiment_utils.get_environment_info()
experiment_utils.print_environment_info(env_info)

ENVIRONMENT INFO
  timestamp: 2026-04-22T04:35:34.480257
  torch_version: 2.10.0+cu128
  cuda_available: True
  cuda_version: 12.8
  device_name: NVIDIA L4
  packages:
    torch: 2.10.0+cu128
    numpy: 2.0.2
    flow_matching: 1.0.10
    POT: 0.9.6.post1
    matplotlib: 3.10.0
    pandas: 2.2.2
    tqdm: 4.67.3


In [6]:
experiment_utils.set_global_seed(GLOBAL_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

[Seed] All random seeds set to 42
Using device: cuda


## GMM Parameters

In [7]:
# ============================================================
# GMM PARAMETERS
# Priority:
#   1. Load from PARAMS_DIR  (shared across runs, committed to repo)
#   2. Load from RESULTS_DIR (fallback from a previous run)
#   3. Generate fresh and save to both dirs
#
# To force regeneration: set FORCE_REGENERATE_PARAMS = True
# ============================================================
FORCE_REGENERATE_PARAMS = False

def _load_params():
    """Try PARAMS_DIR first, then RESULTS_DIR."""
    loaded = experiment_utils.load_gmm_params(PARAMS_DIR, EXPERIMENT_NAME)
    if loaded is not None:
        print(f"[GMM] Loaded from PARAMS_DIR: {PARAMS_DIR}")
        return loaded
    loaded = experiment_utils.load_gmm_params(RESULTS_DIR, EXPERIMENT_NAME)
    if loaded is not None:
        print(f"[GMM] Loaded from RESULTS_DIR: {RESULTS_DIR}")
        return loaded
    return None

loaded = None if FORCE_REGENERATE_PARAMS else _load_params()

if loaded is not None:
    mu_list, Sigma_list, alpha, mog_means, mog_variances, weights, x_star = loaded
    mu_list    = [mu.float() for mu in mu_list]
    Sigma_list = [cov.float() for cov in Sigma_list]
    alpha      = alpha.float()
else:
    print("[GMM] Generating fresh parameters...")
    experiment_utils.set_global_seed(GLOBAL_SEED)   # seed generation for reproducibility
    mu_list, Sigma_list, alpha, mog_means, mog_variances, weights, x_star = \
        dist_utils.get_param_mog_with_target(
            dim_data=5, num_components=4, device='cpu',
            conditional_modes=2, distanceOrScale="Distance"
        )
    mog_means, mog_variances, weights = dist_utils.filter_and_normalize(
        mog_means, mog_variances, weights, threshold=0.001
    )
    mu_list    = [mu.float() for mu in mu_list]
    Sigma_list = [cov.float() for cov in Sigma_list]
    alpha      = alpha.float()

    # save to PARAMS_DIR (canonical, share across runs)
    experiment_utils.save_gmm_params(
        mu_list, Sigma_list, alpha,
        mog_means, mog_variances, weights, x_star,
        PARAMS_DIR, EXPERIMENT_NAME
    )


print(f"x_star = {x_star}")
print(f"Number of conditional modes after filtering: {len(mog_means)}")

[GMM] Parameters loaded from /content/conditional-matching-paper/simulations/params/5D_cond_1D_gmm_params.pt
[GMM] Loaded from PARAMS_DIR: /content/conditional-matching-paper/simulations/params
x_star = tensor([-4.4615, -0.2913, -0.9775, -4.8282])
Number of conditional modes after filtering: 2


## Data

In [8]:
experiment_utils.set_global_seed(GLOBAL_SEED)
X = dist_utils.generate_mog_samples(25_000, mu_list, Sigma_list, alpha).float().to(device)

[Seed] All random seeds set to 42


## Train Models

### Consistency Model — P(Y|X=x)

In [9]:
experiment_utils.set_global_seed(GLOBAL_SEED)

B, C      = X.shape
nfeatures = C - CONDITION_ON
data_generator_cm = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha
)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(
    nfeatures=nfeatures, condition_on=CONDITION_ON,
    nunits=NUNITS_CM, depth=NBLOCKS_CM
)

_loaded_cm = experiment_utils.load_model_checkpoint(
    Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_cm:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    Cos_ConsistencyModeliCT.train_model(
        X=None, nepochs=NEPOCHS_CM, batch_size=BATCH_SIZE_CM,
        device=device, condition=CONDITION_ON,
        data_generator=data_generator_cm, use_improved_training=True
    )
    experiment_utils.save_model_checkpoint(
        Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

[Seed] All random seeds set to 42
[Checkpoint] CM loaded from /content/conditional-matching-paper/simulations/checkpoints/5D_cond_1D/5D_cond_1D_CM_seed42.pt


### Diffusion — P(Y|X=x)

In [10]:
experiment_utils.set_global_seed(GLOBAL_SEED)

X_train   = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)
nfeatures = X_train.shape[1]

data_generator_diff_cond = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha, kernel_func=None
)

model_cond = Diffusion.DiffusionModel(
    nfeatures=nfeatures, nblocks=NBLOCKS, nunits=NUNITS,
    condition=True, condition_on=CONDITION_ON,
    diffusion_steps=DIFFUSION_STEPS
)

_loaded_diff_cond = experiment_utils.load_model_checkpoint(
    model_cond, "Diffusion_cond", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_diff_cond:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    model_cond.train_model(
        None, data_generator=data_generator_diff_cond,
        nepochs=NEPOCHS, batch_size=BATCH_SIZE,
        condition_on=CONDITION_ON
    )
    experiment_utils.save_model_checkpoint(
        model_cond, "Diffusion_cond", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

[Seed] All random seeds set to 42
[Checkpoint] Diffusion_cond loaded from /content/conditional-matching-paper/simulations/checkpoints/5D_cond_1D/5D_cond_1D_Diffusion_cond_seed42.pt


### Diffusion — P(X=x)

In [11]:
experiment_utils.set_global_seed(GLOBAL_SEED)

data_generator_diff_uncond = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha,
    kernel_func=lambda X: X[:, :CONDITION_ON]
)

model_uncond = Diffusion.DiffusionModel(
    nfeatures=CONDITION_ON, nblocks=NBLOCKS, nunits=NUNITS,
    condition=False, diffusion_steps=DIFFUSION_STEPS
)

_loaded_diff_uncond = experiment_utils.load_model_checkpoint(
    model_uncond, "Diffusion_uncond", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_diff_uncond:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    model_uncond.train_model(
        None, data_generator=data_generator_diff_uncond,
        nepochs=NEPOCHS, batch_size=BATCH_SIZE,
        condition_on=CONDITION_ON
    )
    experiment_utils.save_model_checkpoint(
        model_uncond, "Diffusion_uncond", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

[Seed] All random seeds set to 42
[Checkpoint] Diffusion_uncond loaded from /content/conditional-matching-paper/simulations/checkpoints/5D_cond_1D/5D_cond_1D_Diffusion_uncond_seed42.pt


## Optimize

### LGD

In [12]:
# NOTE: optimize_LGD call is untouched — only seed management added around it
best_x_t_LGD_list = []
l2_gmm_LGD_list   = []
l2_x_LGD_list     = []
lgd_times         = []
final_loss_LGD    = []

for i in trange(N_ATTEMP_OPTIM):
    run_seed = experiment_utils.set_run_seed(GLOBAL_SEED, i)

    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, model_cond, mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, loss="MMD", device=device,
        num_x_t=NUM_X_T_LGD
    )
    best_x_t = best_x_t.reshape(-1, 1)
    end_time = time.time()

    lgd_times.append(end_time - start_time)
    final_loss_LGD.append(final_loss)
    best_x_t_LGD_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = dist_utils.gmm_l2_distance(
        mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights
    )
    l2_x = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_list.append(l2_gmm)
    l2_x_LGD_list.append(l2_x)
    print(f"[{i+1}] seed={run_seed} | L2 GMM: {l2_gmm:.6f} | L2 to x*: {l2_x:.6f}")

  0%|          | 0/25 [00:00<?, ?it/s]/content/conditional-matching-paper/simulations/src/dist_utils.py:466: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4480.)
  exponent = -0.5 * diff.T @ Sigma_22_inv @ diff
  4%|▍         | 1/25 [07:29<2:59:44, 449.35s/it]

[1] seed=42 | L2 GMM: 0.159583 | L2 to x*: 26.508577


  8%|▊         | 2/25 [14:50<2:50:27, 444.68s/it]

[2] seed=43 | L2 GMM: 0.543279 | L2 to x*: 97.198746


 12%|█▏        | 3/25 [22:11<2:42:23, 442.87s/it]

...

[23] seed=64 | L2 GMM: 1.014777 | L2 to x*: 66.689247


 96%|█████████▌| 24/25 [2:57:06<07:22, 442.68s/it]

[24] seed=65 | L2 GMM: 0.570904 | L2 to x*: 33.241283


100%|██████████| 25/25 [3:04:30<00:00, 442.83s/it]

[25] seed=66 | L2 GMM: 0.486307 | L2 to x*: 33.453545


### LGD-CM

In [13]:
# NOTE: optimize_LGD call is untouched — only seed management added around it
best_x_t_LGD_CM_list = []
l2_gmm_LGD_CM_list   = []
l2_x_LGD_CM_list     = []
lgd_cm_times         = []
final_loss_LGD_CM    = []

for i in trange(N_ATTEMP_OPTIM):
    run_seed = experiment_utils.set_run_seed(GLOBAL_SEED, i)

    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, Cos_ConsistencyModeliCT,
        mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, loss="MMD", device=device,
        CM=True, FLAG=False, num_x_t=NUM_X_T_LGD_CM
    )
    best_x_t = best_x_t.reshape(-1, 1)
    end_time = time.time()

    lgd_cm_times.append(end_time - start_time)
    final_loss_LGD_CM.append(final_loss)
    best_x_t_LGD_CM_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = dist_utils.gmm_l2_distance(
        mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights
    )
    l2_x = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_CM_list.append(l2_gmm)
    l2_x_LGD_CM_list.append(l2_x)
    print(f"[{i+1}] seed={run_seed} | L2 GMM: {l2_gmm:.6f} | L2 to x*: {l2_x:.6f}")

  4%|▍         | 1/25 [00:22<09:09, 22.91s/it]

[1] seed=42 | L2 GMM: 0.563674 | L2 to x*: 14.539968


  8%|▊         | 2/25 [00:45<08:47, 22.96s/it]

[2] seed=43 | L2 GMM: 0.156538 | L2 to x*: 21.958218


 12%|█▏        | 3/25 [01:08<08:24, 22.94s/it]

...

[23] seed=64 | L2 GMM: 0.156698 | L2 to x*: 29.144663


 96%|█████████▌| 24/25 [09:08<00:22, 22.84s/it]

[24] seed=65 | L2 GMM: 1.217805 | L2 to x*: 69.636276


100%|██████████| 25/25 [09:31<00:00, 22.85s/it]

[25] seed=66 | L2 GMM: 0.159144 | L2 to x*: 25.142717


## Results

In [14]:
rows = [
    experiment_utils.summary_row("LGD",    l2_gmm_LGD_list,    l2_x_LGD_list,    lgd_times),
    experiment_utils.summary_row("LGD-CM", l2_gmm_LGD_CM_list, l2_x_LGD_CM_list, lgd_cm_times),
]
df = pd.DataFrame(rows).set_index("Method")
display(df)

rows_top10 = [
    experiment_utils.top10_stats("LGD",    final_loss_LGD,    l2_gmm_LGD_list,    l2_x_LGD_list,    lgd_times),
    experiment_utils.top10_stats("LGD-CM", final_loss_LGD_CM, l2_gmm_LGD_CM_list, l2_x_LGD_CM_list, lgd_cm_times),
]
df_top10 = pd.DataFrame(rows_top10).set_index("Method")
display(df_top10)

,L2 GMM mean,L2 GMM std,L2 to x* mean,L2 to x* std,Time mean (s),Time std (s)
Method,,,,,,
LGD,0.5325,0.3409,124.5027,331.1374,442.81,1.96
LGD-CM,0.3848,0.3936,28.5157,15.8624,22.84,0.20


,Loss mean,Loss std,L2 GMM mean,L2 GMM std,L2 to x* mean,L2 to x* std,Time mean (s),Time std (s),Top-k selected
Method,,,,,,,,,
LGD,0.2535,0.0900,0.2922,0.1667,27.9117,5.1573,443.02,2.33,10
LGD-CM,0.2672,0.0653,0.4085,0.4173,19.4510,5.7298,22.80,0.17,10


In [15]:
def to_python(val):
    if isinstance(val, torch.Tensor):
        return val.detach().cpu().tolist()
    if isinstance(val, np.ndarray):
        return val.tolist()
    if hasattr(val, "item"):
        return val.item()
    return val

results = {
    "experiment":  EXPERIMENT_NAME,
    "seed":        GLOBAL_SEED,
    "environment": env_info,
    "LGD": {
        "x_pred":     [to_python(x) for x in best_x_t_LGD_list],
        "final_loss": [to_python(l) for l in final_loss_LGD],
        "l2_gmm":     l2_gmm_LGD_list,
        "l2_x":       l2_x_LGD_list,
        "times":      lgd_times,
    },
    "LGD-CM": {
        "x_pred":     [to_python(x) for x in best_x_t_LGD_CM_list],
        "final_loss": [to_python(l) for l in final_loss_LGD_CM],
        "l2_gmm":     l2_gmm_LGD_CM_list,
        "l2_x":       l2_x_LGD_CM_list,
        "times":      lgd_cm_times,
    },
    "meta": {
        "n_attemp_optim":            N_ATTEMP_OPTIM,
        "nsamples_in_optim_for_mmd": NSAMPLES_IN_OPTIM_FOR_MMD,
        "x_star":                    to_python(x_star),
    },
}

path = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_results_seed{GLOBAL_SEED}.json")
with open(path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {path}")

Results saved to /content/conditional-matching-paper/simulations/results/5D_cond_1D/5D_cond_1D_results_seed42.json


In [16]:
from google.colab import files
import zipfile

# 1. Download the Results JSON
print(f"Downloading results: {path}")
files.download(path)

# 2. Zip the Checkpoints directory and download it
zip_path = f"/content/{EXPERIMENT_NAME}_checkpoints.zip"
print(f"Zipping checkpoints to {zip_path}...")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_dir in os.walk(CHECKPOINT_DIR):
        for file in files_in_dir:
            file_full_path = os.path.join(root, file)
            # Store with a relative path inside the zip
            arcname = os.path.relpath(file_full_path, CHECKPOINT_DIR)
            zipf.write(file_full_path, arcname)

print("Downloading checkpoints zip...")
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Zipping checkpoints to /content/5D_cond_1D_checkpoints.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>